# Notebook for LSTM 
Most recent update (02.04.2024): construct a univariate LSTM to implement the lagged approach.

In [7]:
#importing functions and  packages 
import numpy as np
import datetime as dt
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import zscore
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.compose import ColumnTransformer
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from helpers import fix_datetime_format

In [5]:
random_state = 42

## Data Import & Preprocessing

In [27]:
# Data import
data = pd.read_csv("data/final_offshore_data_2017_2025.csv", dtype = {'year_mon_day':'int32'})

#remove 2025 data
data = data[data["full_datetime"] < "2025-01-01-01"]
data.tail()

,year_mon_day,hour,wind_dir_avg_10,wind_speed_h_avg,wind_speed_avg_10,air_pressure,humidity,full_datetime,capacity,volume,percentage,emission,emissionfactor,correct_days
70123,20241231,20,211.999178,146.000000,151.333333,10141.5,86.571429,2024-12-31-20,3718000,3718000,0.894180,0,0,2024-12-31-20
70124,20241231,21,210.666633,146.000000,148.000000,10133.5,86.857143,2024-12-31-21,3744500,3744500,0.900553,0,0,2024-12-31-21
70125,20241231,22,210.000000,146.000000,149.333333,10124.0,88.285714,2024-12-31-22,3686750,3686750,0.886664,0,0,2024-12-31-22
70126,20241231,23,213.336340,150.000000,149.333333,10118.5,88.857143,2024-12-31-23,3719250,3719250,0.894481,0,0,2024-12-31-23
70127,20241231,24,214.005693,152.666667,153.333333,10109.5,87.428571,2024-12-31-24,3453249,3453249,0.830507,0,0,2024-12-31-24


In [28]:
# fix correct_days variable and turn into index
# Apply the fix
data["correct_days"] = data["correct_days"].apply(fix_datetime_format)

#set index
data.set_index('correct_days', inplace=True)
data.tail()

,year_mon_day,hour,wind_dir_avg_10,wind_speed_h_avg,wind_speed_avg_10,air_pressure,humidity,full_datetime,capacity,volume,percentage,emission,emissionfactor
correct_days,,,,,,,,,,,,,
2024-12-31 20:00:00,20241231,20,211.999178,146.000000,151.333333,10141.5,86.571429,2024-12-31-20,3718000,3718000,0.894180,0,0
2024-12-31 21:00:00,20241231,21,210.666633,146.000000,148.000000,10133.5,86.857143,2024-12-31-21,3744500,3744500,0.900553,0,0
2024-12-31 22:00:00,20241231,22,210.000000,146.000000,149.333333,10124.0,88.285714,2024-12-31-22,3686750,3686750,0.886664,0,0
2024-12-31 23:00:00,20241231,23,213.336340,150.000000,149.333333,10118.5,88.857143,2024-12-31-23,3719250,3719250,0.894481,0,0
2025-01-01 00:00:00,20241231,24,214.005693,152.666667,153.333333,10109.5,87.428571,2024-12-31-24,3453249,3453249,0.830507,0,0
